## **CBADVAI Phase 2: Machine Learning**

Following the completion of data preparation and exploratory data analysis, machine learning models were developed to predict Actual Usage Behavior (AUB). This section describes the procedures for data splitting, preprocessing, model training, hyperparameter tuning, and performance evaluation.

In [ ]:
# Code explanation: Import the libraries required for data processing, visualization, and model development.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

import os
import sys

sys.path.append(os.path.abspath(".."))

# Import preprocessing wrapper functions
from scripts.preprocessing import (
    run_preprocessing,
)

from scripts.machinelearning import (
    select_features,
    select_target,
    split_dataset,
    create_kfold,
    standardize_features,
    create_rf_param_grid,
    create_rf_gridsearch,
    fit_rf_gridsearch,
    get_best_rf_params,
    summarize_rf_results,
    get_best_rf_model,
    predict_rf,
    evaluate_regression,
    create_mlp_param_grid,
    create_mlp_gridsearch,
    fit_mlp_gridsearch,
    get_best_mlp_params,
    get_best_mlp_cv_mse,
    get_best_mlp_model,
    predict_mlp,
    compare_models
)

# Set the chart style for visualizations.
plt.style.use('seaborn-v0_8-darkgrid')

# Display plots inline within the notebook.
%matplotlib inline

In [ ]:
scommerce_df = run_preprocessing()

### [5] Feature and Target Variable Selection

This section describes the predictor and target variables used for machine learning model development. The selected variables were based on the conceptual framework of the original study, with the implementation adapted for a machine learning-based prediction task.

In [ ]:
feature_columns = [
    "PU",
    "PEU",
    "FSC",
    "SP",
    "TP",
    "IB"
]

target_column = "AUB"

X = select_features(
    scommerce_df,
    feature_columns
)

y = select_target(
    scommerce_df,
    target_column
)

The predictor variables consisted of Perceived Usefulness (PU), Perceived Ease of Use (PEU), Familiarity with Social Commerce (FSC), Social Participation (SP), Trust in Platform (TP), and Intention to Buy (IB). These constructs were selected as input features since they were identified in the original study as factors influencing Actual Usage Behavior (AUB) within the proposed conceptual framework.

Actual Usage Behavior (AUB) was designated as the target variable because it represents the primary behavioral outcome of interest in the original study. Rather than testing causal relationships through PLS-SEM, this study reformulated the problem as a supervised machine learning task, where the objective was to predict AUB based on the six behavioral and technological constructs.

### [6] Dataset Partitioning

This section describes how the dataset is partitioned into training and testing subsets to support model development and evaluation. It also defines the cross-validation strategy used during hyperparameter optimization.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(
    X,
    y,
    test_size=0.20,
    random_state=1
)

An 80:20 train-test split is adopted, where 80% of the dataset is allocated for model training while the remaining 20% is reserved for final model evaluation. This proportion provides a balance between allowing the models to learn from a sufficient amount of data while preserving an independent test set for assessing generalization performance on unseen observations (TpointTech, 2026).

Because the dataset consists of only 757 observations, a separate validation set is not created. Instead, hyperparameter tuning and model selection are performed using cross-validation on the training data. This approach allows more efficient utilization of the available observations while maintaining an untouched test set for unbiased model evaluation (Prathik, 2025).

The parameter random_state=1 fixes the random seed used during data partitioning, ensuring that the same training and testing subsets are generated whenever the experiment is repeated. This improves the reproducibility of the machine learning pipeline.

#### *Cross Validation*

To support model selection and hyperparameter optimization, a ten-fold cross-validation strategy is defined using the KFold class.

In [ ]:
kf = create_kfold(
    n_splits=10,
    shuffle=True,
    random_state=1
)

The KFold object specifies that the training dataset will be partitioned into ten approximately equal folds. During model training, one fold is used as the validation subset while the remaining nine folds are used for training. This process is repeated ten times so that every fold serves as the validation set exactly once, after which the validation results are averaged to estimate the model's performance.

Unlike the conventional train-validation-test approach, this study employs 10-fold cross-validation on the training data instead of creating a separate validation set. Because the dataset contains 757 observations, this approach allows more efficient utilization of the available training data while preserving an independent test set for the final evaluation. Additionally, every training observation contributes to both model training and validation across different iterations, resulting in a more reliable estimate of model performance than relying on a single train-validation split (Prathik, 2025).

A value of n_splits=10 is selected because 10-fold cross-validation is widely regarded as a standard practice in machine learning, providing a favorable balance between reliable performance estimation and computational efficiency (Brownlee, 2020). Furthermore, the parameter shuffle=True randomly shuffles the training observations before partitioning them into folds, reducing potential bias caused by the original ordering of the dataset.

### [7] Feature Scaling
Feature scaling is performed to transform the predictor variables into a common scale prior to model training. This preprocessing step helps reduce differences in feature magnitudes, allowing machine learning algorithms to learn from the data more effectively.

In [ ]:
(   scaler,
    X_train_scaled,
    X_test_scaled
) = standardize_features(
    X_train,
    X_test
)

Standardization is performed using the StandardScaler class from the scikit-learn library. This technique transforms each feature to have a mean of 0 and a standard deviation of 1, reducing the influence of differences in feature magnitudes while preserving the overall distribution of the data.

Standardization is selected instead of normalization because the MLP is trained using gradient-based optimization algorithms, which generally perform better when input features are centered around zero and have comparable variances. In contrast, Min-Max normalization rescales values to a fixed range, typically between 0 and 1, based on the minimum and maximum values of each feature. Because normalization depends directly on these extreme values, it is generally more sensitive to outliers than standardization. Consequently, standardization is preferred for gradient-based learning algorithms, whereas normalization is more commonly applied to distance-based algorithms such as k-Nearest Neighbors (k-NN).

To prevent data leakage, the scaler is fitted only on the training data using fit_transform(). The learned scaling parameters are subsequently applied to the testing data through transform() without recomputing the feature statistics. This ensures that information from the testing set is not introduced during model training, preserving the validity of the model evaluation.

### **[8] Random Forest**

Random Forest is an ensemble learning algorithm that combines multiple Decision Trees to produce a more accurate and robust prediction. Instead of relying on a single tree, Random Forest aggregates the predictions of many trees, reducing overfitting while improving generalization.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

The Random Forest hyperparameter search was informed by Probst, Wright, and Boulesteix (2019) and the scikit-learn implementation of Random Forest Regression. The selected hyperparameters were chosen to balance established recommendations with the characteristics of the present dataset, which consists of 757 observations and six predictor variables.

* The `n_estimators` values of 100, 200, and 300 were selected to examine whether increasing the number of trees improves model stability and predictive performance. The value of 100 provides a standard baseline, while higher values allow the effect of additional trees to be evaluated, consistent with Probst et al. (2019), who note that increasing the number of trees generally improves stability but provides diminishing returns as the forest becomes larger.

* The `max_features` values of 2, 3, and 6 were included because Probst et al. identify the number of candidate predictors at each split (`mtry`) as an important Random Forest hyperparameter. With six predictors, a value of 2 corresponds to the commonly discussed ($\frac{p}{3}$) setting for regression, while 3 approximates ($\sqrt{p}$), and 6 allows all predictors to be considered at each split.

* The `max_depth` values of None, 10, and 20 were selected to evaluate different levels of tree complexity, ranging from unrestricted growth to progressively constrained trees.

* The `min_samples_split` values of 2 and 5 were included to compare the standard minimum split requirement with a more restrictive setting, while `min_samples_leaf` values of 1, 2, and 5 were selected to evaluate different levels of node-size restriction. The value of 5 is particularly relevant because Probst et al. (2019) discuss a node size of approximately 5 as a typical setting for Random Forest regression.

Overall, these hyperparameters provide a literature-informed search space that allows the model to evaluate different levels of ensemble size, feature randomization, and tree complexity. Grid search with cross-validation was then used to determine which configuration provided the best predictive performance for the present dataset, rather than assuming that any single configuration was universally optimal.

In [ ]:
param_grid_rf = create_rf_param_grid()

"""
Default values in scikit-learn:
  RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
  )
"""

| Hyperparameter       | Description                                                                          |
| -------------------- | ------------------------------------------------------------------------------------ |
| `n_estimators`        | Determines the number of decision trees in the Random Forest.                       |
| `max_features`        | Specifies the number of features considered when determining the best split.        |
| `max_depth`           | Controls the maximum depth of each decision tree.                                   |
| `min_samples_split`   | Specifies the minimum number of samples required to split an internal node.         |
| `min_samples_leaf`    | Specifies the minimum number of samples required in a leaf node.                    |

The code below initializes the GridSearchCV object for the Random Forest Regressor using the predefined hyperparameter search space. The model is configured with a fixed random seed to ensure reproducible results. Hyperparameter combinations are evaluated using the 10-fold cross-validation strategy, with negative mean squared error (MSE) as the optimization metric. Parallel processing is also enabled to evaluate the search space more efficiently.

In [ ]:
grid_rf = create_rf_gridsearch(
    param_grid=param_grid_rf,
    cv=kf,
    random_state=1,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

After initializing the GridSearchCV object, the model fitting process can begin. During training, GridSearchCV systematically evaluates every possible combination of the specified Random Forest hyperparameters using the training data and 10-fold cross-validation.

In [ ]:
grid_rf = fit_rf_gridsearch(
    grid_rf,
    X_train,
    y_train
)

The hyperparameter combination that achieved the best average performance during the GridSearchCV search was displayed, along with its corresponding cross-validation Mean Squared Error (MSE). Since GridSearchCV maximizes the negative MSE score, the value was multiplied by **-1** to obtain the actual MSE, where lower values indicate better predictive performance.

In [ ]:
best_params, best_cv_mse = get_best_rf_params(
    grid_rf
)

print("Best Hyperparameters:")
print(best_params)

print("\nBest Cross-Validation MSE:")
print(best_cv_mse)

The GridSearchCV results were converted into a DataFrame to summarize the performance of each hyperparameter combination. The selected columns display the values of the Random Forest hyperparameters, the mean cross-validation score, and the corresponding ranking.

In [ ]:
results = summarize_rf_results(
    grid_rf
)

results

The output shows the selected hyperparameter values, their mean cross-validation score, and their corresponding rank. Since the scoring metric is negative Mean Squared Error (MSE), values closer to zero indicate better predictive performance. The best-performing configuration was **300 trees, 2 features considered at each split, a maximum depth of 10, a minimum of 5 samples required to split a node, and a minimum of 2 samples per leaf**, which achieved a mean cross-validation score of **-0.162678** and was ranked **1st**. The second- and third-ranked configurations produced very similar scores of **-0.162730** and **-0.162991**, respectively, indicating only small differences in cross-validation performance among the top configurations. In contrast, the lowest-ranked configuration had a score of **-0.181909**, showing poorer cross-validation performance. The results are sorted by **rank_test_score** so that the best-performing configurations appear first.

The best-performing Random Forest model identified by GridSearchCV was retrieved and used to generate predictions on the test dataset. The selected model corresponds to the hyperparameter combination that achieved the best average performance during the cross-validation process, while the test data was used to evaluate its predictive performance on unseen observations.

In [ ]:
best_rf = get_best_rf_model(
    grid_rf
)

rf_pred = predict_rf(
    best_rf,
    X_test
)

The required evaluation metrics were imported to assess the Random Forest model's predictive performance. Mean Absolute Error (MAE), Mean Squared Error (MSE), and R² score were used to evaluate the differences between the actual and predicted values, while NumPy was imported to support numerical calculations.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

The tuned Random Forest model was evaluated on the test set using Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R². The MAE, MSE, and RMSE values measure the magnitude of prediction errors, with lower values indicating better performance, while the R² score measures the proportion of variance in the target variable explained by the model, with higher values indicating better predictive performance. The resulting metrics provide an overall assessment of the model's predictive performance on unseen test data.

In [ ]:
print("Tuned Random Forest Results")

rf_mae, rf_mse, rf_rmse, rf_r2 = evaluate_regression(
    y_test,
    rf_pred
)

print("\nEvaluation on Test Set")
print(f"MAE : {rf_mae:.4f}")
print(f"MSE : {rf_mse:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R²  : {rf_r2:.4f}")

#### **Interpretation of Metrics**

The optimized Random Forest model achieved a **Mean Absolute Error (MAE)** of **0.2304**, indicating that its predictions differed from the actual AUB values by an average of approximately **0.23** on the five-point Likert scale. This relatively low error suggests that the model is generally able to predict respondents' Actual Usage Behavior with good accuracy.

The model also obtained a **Mean Squared Error (MSE)** of **0.1304** and a **Root Mean Squared Error (RMSE)** of **0.3611**. Since RMSE is expressed in the same unit as the target variable, it indicates that the model's predictions typically deviate from the actual AUB values by approximately **0.36 points**. The difference between the MAE and RMSE is relatively small, suggesting that although some larger prediction errors may be present, they do not substantially increase the model's overall error.

Furthermore, the optimized Random Forest achieved an **R² score of 0.7520**, indicating that approximately **75.2%** of the variation in Actual Usage Behavior can be explained by the predictor variables included in the model. This suggests that the behavioral and perception constructs collectively provide strong predictive information for estimating AUB, with the model accounting for a substantial proportion of the observed variation in the target variable.

In conclusion, the evaluation metrics indicate that the tuned Random Forest model produces accurate predictions while maintaining good generalization on unseen data. The relatively low MAE and RMSE, together with the **R² value of 0.7520**, demonstrate that the model effectively captures the relationship between the behavioral constructs and Actual Usage Behavior. Although some prediction error remains, the model explains a substantial proportion of the variation in AUB and demonstrates strong predictive performance on the test set.

The actual and predicted AUB values were visualized using a scatter plot to assess the Random Forest model's prediction performance. The **actual values** are plotted on the x-axis, while the **predicted values** are plotted on the y-axis. The dashed reference line represents perfect predictions, where the predicted value is equal to the actual value. Points located closer to this line indicate more accurate predictions, while greater deviations from the line represent larger prediction errors.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(
    figsize=(8, 6)          # Set the figure size
)

plt.scatter(
    y_test,                 # Actual target values
    rf_pred,                # Predicted target values
    alpha=0.6               # Set point transparency
)

plt.plot(
    [y_test.min(), y_test.max()],    # Minimum and maximum actual values (x-axis)
    [y_test.min(), y_test.max()],    # Minimum and maximum actual values (y-axis)
    linestyle="--"                   # Reference line representing perfect predictions
)

plt.xlabel("Actual Values")          # Label the x-axis
plt.ylabel("Predicted Values")       # Label the y-axis
plt.title("Random Forest: Actual vs Predicted")  # Set the plot title

plt.tight_layout()                   # Adjust spacing to prevent overlapping elements
plt.show()                           # Display the plot

#### **Interpretation of Scatter Plot**

The scatter plot compares the **actual AUB values** with the **predicted AUB values** generated by the optimized Random Forest model. The dashed diagonal line represents perfect predictions, where the predicted value is equal to the actual value. Overall, the points show a clear positive relationship and are generally concentrated around the reference line, indicating that the model is able to capture the overall pattern of Actual Usage Behavior.

The predictions are particularly well aligned with the actual values at the lower and upper ends of the scale. For example, observations with actual values close to **1.0 and 5.0** generally have predictions that are relatively close to their corresponding actual values. This suggests that the model is able to distinguish respondents with relatively low and high AUB values reasonably well.

However, greater dispersion can be observed around the middle-to-upper portion of the scale, particularly for actual values between approximately **3.0 and 4.5**. Several observations fall noticeably above or below the reference line, indicating that the model produces larger prediction errors for some observations in this range. The clustering of points around actual values of **3.0, 3.5, and 4.0** also suggests that predictions are concentrated around these common response levels.

Overall, the scatter plot indicates **reasonably strong agreement between the actual and predicted values**, as reflected by the general concentration of observations around the diagonal reference line. This visual pattern is consistent with the model's **R² of 0.7520**, which indicates that the Random Forest explains approximately **75.2% of the variation in AUB** on the test set. Nevertheless, the dispersion of some points away from the reference line demonstrates that the model does not perfectly predict every observation, which is also reflected in the **MAE of 0.2304** and **RMSE of 0.3611**.

To further assess the prediction performance of the optimized Random Forest model, the residuals were calculated as the difference between the actual and predicted AUB values and plotted against the predicted values. The dashed horizontal line at **zero** represents perfect predictions, where the actual and predicted values are equal. Residuals close to zero indicate smaller prediction errors, while larger positive or negative residuals indicate greater underprediction or overprediction, respectively.

In [ ]:
rf_residuals = y_test - rf_pred    # Compute the residuals (actual - predicted)

plt.figure(
    figsize=(8, 6)                 # Set the figure size
)

plt.scatter(
    rf_pred,                       # Predicted target values
    rf_residuals,                  # Residual values
    alpha=0.6                      # Set point transparency
)

plt.axhline(
    y=0,                           # Reference line indicating zero residual
    linestyle="--"
)

plt.xlabel("Predicted Values")     # Label the x-axis
plt.ylabel("Residuals")            # Label the y-axis
plt.title("Random Forest: Residual Plot")  # Set the plot title

plt.tight_layout()                 # Adjust spacing to prevent overlapping elements
plt.show()                         # Display the plot

#### **Interpretation of Residual Plot**

The residual plot was examined to further assess the prediction errors of the optimized Random Forest model and determine whether any systematic patterns are present. The residuals are generally distributed on both sides of the **zero reference line**, indicating that the model produces both underpredictions and overpredictions rather than consistently making errors in only one direction.

A considerable portion of the residuals is concentrated relatively close to zero, particularly for predicted values between approximately **3.0 and 4.5**. This indicates that many of the model's predictions have relatively small errors. However, some observations show larger positive and negative residuals, with residuals reaching approximately **+1.3 and −1.3**. These observations represent cases where the predicted values differ more substantially from the actual AUB values.

The residuals also appear to become more dispersed as the predicted values increase, particularly around predicted values between **3.5 and 4.5**. This indicates that the magnitude of the prediction errors is not completely uniform across the range of predicted values. Nevertheless, the points remain distributed around the zero line without an obvious strong systematic pattern or consistent curvature.

Overall, the residual plot suggests that the optimized Random Forest model does not exhibit a strong directional bias in its predictions, as both positive and negative residuals are present. Most predictions remain reasonably close to the zero-error line, which is consistent with the relatively low **MAE of 0.2304** and **RMSE of 0.3611** observed on the test set. The presence of several larger residuals indicates that some observations are more difficult for the model to predict accurately, but these errors do not appear to dominate the model's overall predictive performance.

As the final visualization for the optimized Random Forest model, the feature importance scores were calculated and visualized to identify the predictor variables that contributed most to the model's predictions. Higher feature importance scores indicate a greater relative contribution to the model's decision-making process. The features were sorted in descending order of importance, allowing the most influential predictors to be identified and compared more easily.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,                  # Feature names
    "Importance": best_rf.feature_importances_   # Feature importance scores from the Random Forest model
})

feature_importance = feature_importance.sort_values(
    by="Importance",                             # Sort by feature importance
    ascending=False                              # Display the most important features first
)

plt.figure(
    figsize=(10, 6)                              # Set the figure size
)

plt.barh(
    feature_importance["Feature"],               # Feature names on the y-axis
    feature_importance["Importance"]             # Importance scores on the x-axis
)

plt.gca().invert_yaxis()                         # Display the most important feature at the top

plt.xlabel("Importance")                         # Label the x-axis
plt.ylabel("Feature")                            # Label the y-axis
plt.title("Random Forest: Feature Importance")   # Set the plot title

plt.tight_layout()                               # Adjust spacing to prevent overlapping elements
plt.show()                                       # Display the plot

### **Interpretation of Feature Importance**

The feature importance analysis indicates that **Perceived Ease of Use (PEU)** was the most influential predictor in the optimized Random Forest model, with an importance score of approximately **0.27**. This suggests that Perceived Ease of Use contributed the most to the model's predictive decisions among the predictor variables included in the analysis.

The second most influential feature was **Perceived Usefulness (PU)**, with an importance score of approximately **0.25**. Its relatively high importance, together with Perceived Ease of Use, indicates that these two perception-related constructs made the largest contributions to predicting **Actual Usage Behavior (AUB)** in the Random Forest model.

**Familiarity with Social Commerce (FSC)** had the third-highest importance at approximately **0.19**, indicating a meaningful contribution to the model's predictions, although its contribution was lower than those of Perceived Ease of Use and Perceived Usefulness. **Social Participant (SP)** followed with an importance score of approximately **0.11**, while **Interaction Behavior (IB)** and **Trust in Platform (TP)** had comparatively lower importance scores of approximately **0.09** and **0.08**, respectively.

Overall, the feature importance results show that the model relied most heavily on **Perceived Ease of Use (PEU)** and **Perceived Usefulness (PU)**, which together account for approximately **52% of the total feature importance**. **Familiarity with Social Commerce (FSC)** also made a substantial contribution, while **Social Participant (SP)**, **Interaction Behavior (IB)**, and **Trust in Platform (TP)** had smaller relative contributions. These results suggest that perceptions related to the **ease of use, usefulness, and familiarity with social commerce** were the most influential predictors of Actual Usage Behavior within the Random Forest model.

It is important to note that feature importance reflects the **relative contribution of each predictor to the Random Forest's predictive decisions** and does not by itself establish a causal relationship between the predictors and Actual Usage Behavior.

### **[9] MLP Regressor**

The Multilayer Perceptron (MLP) Regressor is a feedforward artificial neural network designed to model complex nonlinear relationships in structured tabular data. Unlike Random Forest, which learns patterns through an ensemble of decision trees, the MLP learns nonlinear relationships using interconnected neurons, hidden layers, and nonlinear activation functions. Including both models allows the study to compare two fundamentally different machine learning approaches and determine which is more suitable for predicting Actual Usage Behavior (AUB).

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

The MLP hyperparameter search was guided by the scikit-learn implementation of MLPRegressor and established neural network practices (Pedregosa et al., 2011; scikit-learn developers, 2025). The selected search space was designed to balance model complexity with the characteristics of the present dataset, which consists of 757 observations and six predictor variables.

*   The `hidden_layer_sizes` configurations of (50), (100), and (100, 50) were selected because the optimal network architecture is problem-dependent and is typically determined experimentally rather than through a fixed rule (Goodfellow et al., 2016; scikit-learn developers, 2025). Given the relatively small dataset, small-to-moderate network architectures were preferred to provide sufficient learning capacity while reducing the risk of overfitting. Including both single- and two-hidden-layer architectures also allows the model to evaluate whether additional network depth improves predictive performance.
*   The `activation` functions ReLU and tanh were evaluated because they are among the most widely used activation functions for feedforward neural networks (Goodfellow et al., 2016). ReLU generally provides faster and more stable optimization, while tanh serves as a suitable alternative for standardized input features due to its zero-centered output (Nwankpa et al., 2018).
* The `learning_rate_init` values of 0.001 and 0.01 were selected to compare two commonly used learning rates that balance convergence speed and training stability during optimization (Goodfellow et al., 2016). Likewise, the `alpha` values of 0.0001 and 0.001 were included to evaluate different levels of L2 regularization, helping reduce overfitting while maintaining model flexibility (scikit-learn developers, 2025).

* Other hyperparameters, such as the `solver`, maximum iterations (`max_iter`), `batch size`, and `early stopping`, were kept fixed and were not included in the GridSearchCV search space. This allowed the optimization to focus on the most influential hyperparameters while keeping the search computationally manageable.

Overall, these hyperparameters provide a literature-informed search space that allows the model to evaluate different network architectures, activation functions, learning rates, and regularization strengths. Grid search with cross-validation was then used to determine which configuration produced the best predictive performance for the present dataset rather than assuming that any single configuration was universally optimal.

In [ ]:
param_grid = create_mlp_param_grid()

"""
Default values in scikit-learn:
  MLPRegressor(
    hidden_layer_sizes=(100,),
    activation="relu",
    learning_rate_init=0.001,
    alpha=0.0001,
    ...
)
"""

| Hyperparameter       | Description                                                                           |
| -------------------- | ------------------------------------------------------------------------------------- |
| `hidden_layer_sizes` | Defines the architecture of the neural network (number of neurons and hidden layers). |
| `activation`         | Specifies the activation function used by neurons.                                    |
| `learning_rate_init` | Determines how quickly the model updates its weights during training.                 |
| `alpha`              | L2 regularization parameter that helps reduce overfitting.                            |


The code below initializes the GridSearchCV object for the MLP Regressor using the predefined hyperparameter search space. The model is configured with a maximum of 1,000 training iterations and a fixed random seed to ensure reproducible results. Hyperparameter combinations are evaluated using the same 10-fold cross-validation strategy applied to the Random Forest model, with negative mean squared error (MSE) as the optimization metric. Parallel processing is also enabled to evaluate the search space more efficiently.

In [ ]:
grid_mlp = create_mlp_gridsearch(
    param_grid=param_grid,
    cv=kf
)

After initializing the GridSearchCV object, the model fitting process can begin. During training, GridSearchCV systematically evaluates every possible combination of the specified hyperparameters.

In [ ]:
grid_mlp = fit_mlp_gridsearch(
    grid_mlp,
    X_train_scaled,
    y_train
)

Display the hyperparameter combination that achieved the best average performance during the GridSearchCV search, then report the corresponding cross-validation Mean Squared Error (MSE). Since GridSearchCV maximizes the negative MSE score, the value is multiplied by **-1** to obtain the actual MSE, where lower values indicate better predictive performance.

In [ ]:
print("Best Hyperparameters:")
print(get_best_mlp_params(grid_mlp))

print("\nBest Cross-Validation MSE:")
print(get_best_mlp_cv_mse(grid_mlp))

The results show that GridSearchCV selected a network with a **single hidden layer containing 100 neurons**, using the **tanh** activation function, a **learning rate of 0.001**, and an **L2 regularization strength (alpha) of 0.001**. This suggests that, among the evaluated configurations, a relatively simple network architecture was sufficient for the dataset while providing the best balance between learning capacity and generalization. The selection of **tanh** indicates that it performed better than ReLU for the standardized input features, while the smaller learning rate and regularization value provided stable training and helped reduce overfitting. This configuration achieved the lowest average cross-validation MSE of **0.1555**, making it the optimal MLP model for the subsequent evaluation.

The optimized MLP model is retrieved using `best_estimator_`, which returns the best hyperparameter combination result from earlier. The model is then used to generate predicted Actual Usage Behavior (AUB) values for the standardized test data. Because the test set was not used during model training or hyperparameter tuning, these predictions provide an unbiased basis for evaluating the model's predictive performance.

In [ ]:
best_mlp = get_best_mlp_model(grid_mlp)

mlp_pred = predict_mlp(
    best_mlp,
    X_test_scaled
)

We then evaluate the predictive performance of the optimized MLP model on the test dataset using four regression metrics: Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and the coefficient of determination (R²). These metrics provide a comprehensive assessment of prediction accuracy and are the same evaluation measures used for the Random Forest model, allowing a fair comparison between the two approaches.

In [ ]:
mlp_mae, mlp_mse, mlp_rmse, mlp_r2 = evaluate_regression(
    y_test,
    mlp_pred
)             # Compute the coefficient of determination (R²)

print("Tuned MLP Results")

print("\nEvaluation on Test Set")
print(f"MAE : {mlp_mae:.4f}")
print(f"MSE : {mlp_mse:.4f}")
print(f"RMSE: {mlp_rmse:.4f}")
print(f"R²  : {mlp_r2:.4f}")

#### Interpretation of Metrics

The optimized MLP model achieved a **Mean Absolute Error (MAE)** of **0.2627**, indicating that its predictions differed from the actual AUB values by an average of approximately **0.26** on the five-point Likert scale. This relatively small error suggests that the model is generally able to predict respondents' Actual Usage Behavior with good accuracy.

The model also obtained a **Mean Squared Error (MSE)** of **0.1526** and a **Root Mean Squared Error (RMSE)** of **0.3906**. Since RMSE is expressed in the same unit as the target variable, it indicates that the model's predictions typically deviate from the actual AUB values by less than **0.4 points**. The difference between the MAE and RMSE is also relatively small, suggesting that while some larger prediction errors exist, they are not frequent enough to substantially affect the model's overall performance.

Furthermore, the optimized MLP achieved an **R² score of 0.7097**, indicating that approximately **71.0%** of the variation in Actual Usage Behavior can be explained by the predictor variables included in the model. This suggests that the behavioral and perception constructs collectively provide strong predictive information for estimating AUB.

In conclusion, the evaluation metrics indicate that the tuned MLP model produces accurate and reliable predictions while maintaining good generalization on unseen data. Although some prediction error remains, the relatively low error values and high coefficient of determination demonstrate that the model effectively captures the relationship between the behavioral constructs and Actual Usage Behavior.

To better visualize the predictive performance of the optimized MLP model, a scatter plot of the actual and predicted AUB values is generated. The dashed red line represents perfect predictions, allowing the agreement between the actual and predicted values to be visually assessed.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(
    figsize=(7, 6)                  # Set the figure size
)

plt.scatter(
    y_test,                         # Actual target values
    mlp_pred,                       # Predicted target values
    alpha=0.7                       # Set point transparency
)

plt.plot(
    [y_test.min(), y_test.max()],   # Minimum and maximum actual values (x-axis)
    [y_test.min(), y_test.max()],   # Minimum and maximum actual values (y-axis)
    'r--'                           # Reference line representing perfect predictions
)

plt.xlabel("Actual Values")         # Label the x-axis
plt.ylabel("Predicted Values")      # Label the y-axis
plt.title("MLP: Actual vs Predicted")  # Set the plot title

plt.grid(True)                      # Display gridlines

plt.tight_layout()                  # Adjust spacing to prevent overlapping elements
plt.show()                          # Display the plot

#### Interpretation of Scatter Plot

Overall, most observations are distributed close to this reference line, indicating that the model is able to predict Actual Usage Behavior with reasonably good accuracy.

The highest concentration of points occurs between AUB values of approximately **3.0 and 4.5**, which corresponds to the range where most respondents are located. Within this region, the predicted values generally follow the upward trend of the actual values, suggesting that the model successfully captures the underlying relationship between the behavioral constructs and Actual Usage Behavior.

Although the predictions generally align with the reference line, some deviations are evident. Several observations fall above the line, indicating that the model overestimated the actual AUB values, while others fall below the line, indicating underestimation. These deviations become slightly more noticeable toward the higher AUB values (around **4.0–5.0**), where a few predictions are farther from the reference line. However, no systematic pattern of overprediction or underprediction is observed, suggesting that the prediction errors are relatively balanced rather than consistently biased in one direction.

The spread of the points around the reference line is relatively small compared with the overall range of the target variable, indicating that the prediction errors are generally moderate. This visual assessment is consistent with the numerical evaluation metrics, particularly the **RMSE of 0.3906** and the **R² value of 0.7097**, which indicate that the optimized MLP model explains approximately **71%** of the variation in Actual Usage Behavior while maintaining relatively low prediction error on the unseen test data. Overall, the scatter plot supports the conclusion that the tuned MLP model generalizes well and provides reliable predictions for this dataset.

To further evaluate the predictive performance of the optimized MLP model, a residual plot is generated. The plot shows the residuals (actual − predicted) against the predicted values. Ideally, the residuals should be randomly scattered around the zero reference line, indicating that the prediction errors are random rather than systematic.

In [ ]:
mlp_residuals = y_test - mlp_pred    # Compute the residuals (actual - predicted)

plt.figure(
    figsize=(7, 5)                   # Set the figure size
)

plt.scatter(
    mlp_pred,                        # Predicted target values
    mlp_residuals,                   # Residual values
    alpha=0.7                        # Set point transparency
)

plt.axhline(
    y=0,                             # Reference line indicating zero residual
    color="red",
    linestyle="--"
)

plt.xlabel("Predicted Values")       # Label the x-axis
plt.ylabel("Residuals")              # Label the y-axis
plt.title("MLP: Residual Plot")      # Set the plot title

plt.grid(True)                       # Display gridlines

plt.tight_layout()                   # Adjust spacing to prevent overlapping elements
plt.show()                           # Display the plot

#### Interpretation of Residual Plot

The residual plot shows that the prediction errors are generally scattered around the zero reference line, indicating that the optimized MLP model does not consistently overestimate or underestimate Actual Usage Behavior. Most residuals are relatively close to zero, suggesting that the model produces reasonably accurate predictions across the test dataset.

No clear trend, curved pattern, or funnel-shaped distribution is observed, indicating that the residuals are randomly distributed rather than systematic. This suggests that the model has successfully captured most of the underlying relationship between the predictor variables and Actual Usage Behavior. Although a few larger residuals are present, particularly for some lower predicted values, these appear to be isolated observations rather than evidence of model bias.

The slightly larger residuals observed for some lower predicted AUB values may be related to the relatively small number of respondents with low engagement levels in the dataset. Because fewer training examples are available in this range, the model has less information from which to learn these patterns, resulting in slightly larger prediction errors for some observations.

Overall, the residual plot supports the evaluation metrics obtained earlier, indicating that the optimized MLP model provides stable predictions and generalizes well to unseen data.

### [10] Comparison of Results between Random Forest and Multi-Layer Perceptron

In [ ]:
comparison = compare_models(
    rf_mae,
    rf_mse,
    rf_rmse,
    rf_r2,
    mlp_mae,
    mlp_mse,
    mlp_rmse,
    mlp_r2
)

comparison

Based on the evaluation metrics summarized in the table above, the tuned Random Forest model outperformed the tuned MLP model across every metric considered. Random Forest achieved a lower **MAE** (0.2304 vs. 0.2627), lower **MSE** (0.1304 vs. 0.1526), and lower **RMSE** (0.3611 vs. 0.3906), indicating that its predictions were, on average, closer to the actual AUB values than those of the MLP. Random Forest also achieved a higher **R²** score (0.7520 vs. 0.7097), meaning it explained approximately **75.2%** of the variance in Actual Usage Behavior compared to **71.0%** for the MLP, which is a difference of roughly 4 percentage points.

Although Random Forest performed better on every metric, the gap between the two models is relatively modest. Both models achieved R² scores above 0.70 and RMSE values below 0.4 on a five-point Likert scale, indicating that both approaches were able to capture the underlying relationship between the perception/behavioral constructs and Actual Usage Behavior reasonably well. This is not entirely surprising given the size of the dataset (757 observations) and the modest number of predictors (six constructs): tree-based ensemble methods such as Random Forest are generally well-suited to smaller, low-dimensional tabular datasets, whereas neural network models like MLP typically benefit from larger sample sizes and more complex feature interactions to fully leverage their flexibility.

Beyond predictive accuracy, Random Forest also offers a practical advantage in this context as it provides directly interpretable feature importance scores (as shown in the earlier analysis), allowing the relative contribution of PU, PEU, FSC, SP, TP, and IB to be examined. The MLP, in contrast, functions largely as a black box, which limits its usefulness for explaining why certain constructs matter more than others. Taken together, these results suggest that Random Forest is the more suitable model for this dataset, both in terms of predictive performance and interpretability.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ["MAE", "MSE", "RMSE", "R²"]    # Evaluation metrics to compare

x = np.arange(len(metrics))               # Generate x-axis positions
width = 0.35                              # Set the width of each bar

plt.figure(
    figsize=(8, 6)                        # Set the figure size
)

rf_bars = plt.bar(
    x - width / 2,                        # Position bars for the Random Forest model
    comparison.loc[0, metrics],           # Random Forest metric values
    width,
    label="Random Forest",
    color="#3B82F6",                      # Blue
)

mlp_bars = plt.bar(
    x + width / 2,                        # Position bars for the MLP model
    comparison.loc[1, metrics],           # MLP metric values
    width,
    label="MLP",
    color="#EF4444",                      # Red
)

plt.bar_label(
    rf_bars,                              # Display Random Forest metric values
    fmt="%.3f",
    padding=3
)

plt.bar_label(
    mlp_bars,                             # Display MLP metric values
    fmt="%.3f",
    padding=3
)

plt.xticks(
    x,
    metrics                               # Set x-axis labels
)

plt.ylabel("Metric Value")                # Label the y-axis
plt.title("Performance Comparison of Random Forest and MLP")  # Set the plot title

plt.legend()                              # Display the legend

plt.tight_layout()                        # Adjust spacing to prevent overlapping elements
plt.show()                                # Display the plot

#### Interpretation of Comparison Chart

The bar chart clearly shows that Random Forest (blue) outperforms MLP (red) in R² and has lower values for MAE, MSE, and RMSE. While the differences in the error metrics are small, the gap in R² is more noticeable. This indicates that Random Forest has a slight but consistent advantage over MLP for predicting Actual Usage Behavior, making both models suitable choices.


### **[11] Recommendation**

Based on the results of this study, the following recommendations are proposed:

**Model Selection.** Given its stronger and more consistent performance across all four evaluation metrics, along with its built-in interpretability through feature importance scores, the Random Forest model is recommended as the preferred model for predicting Actual Usage Behavior (AUB) in this context. The MLP remains a reasonable alternative, particularly if future research includes a larger dataset or more complex, non-linear feature interactions that could better leverage its flexibility.

**Platform Design Implications.** The feature importance results indicate that Perceived Ease of Use (PEU) and Perceived Usefulness (PU) were the strongest predictors of Actual Usage Behavior, together accounting for roughly half of the model's total feature importance. This suggests that social commerce platforms targeting Generation Z university students in Vietnam should prioritize improving the ease of navigation, usability, and perceived functional value of their platforms, as these factors appear to have the greatest influence on driving actual engagement, more so than trust or social participation alone.

**Future Research.** Because the dataset was restricted to a specific age range (18–27) and geographic context (Vietnam), the findings should not be generalized beyond this population. Future studies could extend this work by (1) collecting a larger and more diverse sample to improve model generalizability and give the MLP more data to learn from, (2) incorporating additional predictor variables (e.g., platform type, purchase frequency, or specific product categories), and (3) testing additional algorithms.

Expanding the dataset in both directions, more observations and more predictors, would give both models more information to learn from during training. A larger sample would especially benefit the MLP, since neural networks are generally data-hungry and tend to improve substantially with more training examples, whereas Random Forest already performs reasonably well even on a comparatively small dataset such as this one. Likewise, adding predictors beyond the six psychological/behavioral constructs used here, such as platform type, purchase frequency, or purchase amount, would provide the models with a more complete picture of the factors driving Actual Usage Behavior, rather than relying solely on attitudinal and perception-based variables. Together, these additions would likely improve both models' predictive performance and their ability to generalize to the broader population.

## References

- Brownlee, J. (2020, August 26). *How to configure K-fold cross-validation*. Machine Learning Mastery. https://machinelearningmastery.com/how-to-configure-k-fold-cross-validation/

- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep learning*. MIT Press. https://www.deeplearningbook.org/

- Nwankpa, C., Ijomah, W., Gachagan, A., & Marshall, S. (2018). *Activation functions: Comparison of trends in practice and research for deep learning*. arXiv. https://arxiv.org/abs/1811.03378

- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

- Prathik, C. (2025, September). *K-fold cross validation: The gold standard for model evaluation*. Medium. https://medium.com/@prathik.codes/k-fold-cross-validation-the-gold-standard-for-model-evaluation-90648b3f6b08

- Probst, P., Wright, M. N., & Boulesteix, A.-L. (2019). Hyperparameters and tuning strategies for random forest. *WIREs Data Mining and Knowledge Discovery, 9*(3), e1301. https://doi.org/10.1002/widm.1301

- Scikit-learn developers. (n.d.). *3.2. Tuning the hyper-parameters of an estimator*. *Scikit-learn User Guide*. Retrieved August 5, 2026, from https://scikit-learn.org/stable/modules/grid_search.html

- Scikit-learn developers. (2025). *MLPRegressor*. *Scikit-learn documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html

- Shaibu, S. (2024, October 15). *Normalization vs. standardization: How to know the difference*. DataCamp. https://www.datacamp.com/tutorial/normalization-vs-standardization

- Tpoint Tech. (2026, April 16). *Why we use an 80–20 split for training and test data*. https://www.tpointtech.com/why-we-use-an-80-20-split-for-training-and-test-data